#### Задача 1 (10 баллов). 

Попробуем себя в решении задачи определения темы текста. Будем считать, что два текста похожи по теме, если у них больше общих слов (только не предлогов с союзами), чем у других текстов. У нашей программы для определения темы будет несколько готовых текстов (достаточно больших!) с уже известной темой в базе: выберите тексты (и темы) самостоятельно, 5-6 будет достаточно. 

Что должна делать программа? При запуске вы ей сообщаете название нового файла с текстом, который нужно классифицировать, она его открывает, обрабатывает и сравнивает с текстами в своей базе. С которым из текстов оказалось больше всего общих слов, того и тема! Очевидно, вам понадобится какие-то слова из текстов отбрасывать (подумайте, каким образом это сделать - здесь на самом деле несколько вариантов концепций), а еще лемматизировать или хотя бы использовать стемминг. 

Когда будете сдавать это задание, пожалуйста, пришлите и файлы с текстами. И имейте в виду, если тексты будут вставлены прямо в код и слишком короткие, я задачу засчитаю только вполовину. 

Напоминаю, как открываются файлы:

```
with open('путь к файлу - пишите прямые слеши', 'r', encoding='utf-8') as f:
    text = f.read() # все содержимое вашего файла считается в одну длинную строку
```

Настоятельно советую оформить код хотя бы в функции. 

In [131]:
# импортируем необходимые библиотеки
import nltk
from nltk.corpus import stopwords 
stop_words = set(stopwords.words('russian')) # для списка стоп-слов

from pymystem3 import Mystem # для лемматизации
m = Mystem()

import string # для удаления пунктуации

In [132]:
def filter(file_path): # функция для обработки текста
    with open(file_path, 'r', encoding='maccyrillic') as file: # открываем файл с текстом
        text = file.read().lower().translate(str.maketrans('', '', string.punctuation)) # приводим к нижнему регистру, убираем пунктуацию

    words_filtered = [] # убираем стоп-слова
    for word in text.split():
        if word.lower() not in stop_words:
            words_filtered.append(word)
    
    words_lemmatized = [] # лемматизируем
    for word in words_filtered:
        lemmas = m.lemmatize(word)
        words_lemmatized.extend(lemmas)
        elem_to_delete = '\n'
        words_lemmatized = [lemma for lemma in words_lemmatized if lemma != elem_to_delete]
    
    return set(words_lemmatized) # получаем уникальные слова

In [133]:
unique_cinema = filter('/Users/katyamazurina/Desktop/texts/cinema.txt')
unique_books = filter('/Users/katyamazurina/Desktop/texts/books.txt')
unique_sport = filter('/Users/katyamazurina/Desktop/texts/sport.txt')
unique_cosmos = filter('/Users/katyamazurina/Desktop/texts/cosmos.txt')
unique_games = filter('/Users/katyamazurina/Desktop/texts/video_games.txt')

In [134]:
# словарь с темами и их уникальными словами
themes = {
    "Книги": unique_books,
    "Кино": unique_cinema,
    "Космос": unique_cosmos,
    "Спорт": unique_sport,
    "Видео-игры": unique_games
}

In [155]:
def find_theme(new_path, themes): # функция определения темы текста
    new_text = filter(new_path) # обрабатываем новый текст
    matches = {} # словарь для совпадающей темы, где ключ - название, а значение - количество общих слов
    for theme, unique_words in themes.items():
        common_words = new_text & unique_words # общие слова - пересечение уникальных слов нового текста с уникальными словами какой-либо темы
        matches[theme] = len(common_words)
        print(f'Количеств общих слов с темой "{theme}" : {matches[theme]}') # выведем количество общих слов со всеми темами
    final_match = max(matches, key=matches.get)
    return final_match 

In [156]:
path = ('/Users/katyamazurina/Desktop/texts/find_theme.txt') # текст про книги, но тема определилась как видео-игры
theme = find_theme(path, themes)
print(f'Предположительно, тема текста - "{theme}"')

Количеств общих слов с темой "Книги" : 85
Количеств общих слов с темой "Кино" : 87
Количеств общих слов с темой "Космос" : 49
Количеств общих слов с темой "Спорт" : 75
Количеств общих слов с темой "Видео-игры" : 90
Предположительно, тема текста - "Видео-игры"


In [157]:
path_2 = ('/Users/katyamazurina/Desktop/texts/new_text.txt') # ещё один текст про книги, тема определилась верно 
theme = find_theme(path_2, themes)
print(f'Предположительно, тема текста - "{theme}"')

Количеств общих слов с темой "Книги" : 117
Количеств общих слов с темой "Кино" : 96
Количеств общих слов с темой "Космос" : 72
Количеств общих слов с темой "Спорт" : 87
Количеств общих слов с темой "Видео-игры" : 107
Предположительно, тема текста - "Книги"


#### Задача 2 (10 баллов). 

Некоторые предлоги в русском языке могут управлять разными падежами (например, "я еду в Лондон" vs "я живу в Лондоне"). Давайте проанализируем эти предлоги и их падежи. Необходимо:

- составить список таких предлогов (РГ-80 вам в помощь)
- взять достаточно большой текст (можно большое художественное произведение)
- сделать морфоразбор этого текста
- Посчитать, как часто и какие падежи встречаются у слова, идущего после предлога.

Примечания: во-первых, имейте в виду, что иногда после предлога могут идти самые неожиданные вещи: "я что, должен ехать на, черт побери, северный полюс?". Во-вторых, неплохо бы учитывать отсутствие пунктуации (конечно, в норме, как нам кажется, предлог обязательно требует зависимое, но! "да иди ты на!") Эти штуки можно отсеять, если просто учитывать только заранее определенные падежи, а не считать все, какие встретились (так и None можно огрести). 

Если будете использовать RNNMorph, возможно, понадобится регулярное выражение и немного терпения. 

In [158]:
from pymorphy2 import MorphAnalyzer
from collections import defaultdict
from nltk.tokenize import word_tokenize

morph = MorphAnalyzer()

In [159]:
preps = {
    'в': {'accs', 'loct'},
    'на': {'accs', 'loct'},
    'за': {'accs', 'ablt'},
    'под': {'accs', 'ablt'},
    'с': {'gent', 'ablt', 'accs'},
    'о': {'accs', 'loct'},
    'об': {'accs', 'loct'},
    'по': {'datv', 'accs', 'loct'},
    'к': {'datv', 'loct'},
    'среди': {'gent', 'accs'},
    'между': {'ablt', 'gent'},
    'над': {'ablt', 'accs'},
    'перед': {'ablt', 'accs'},
    'из-за': {'gent', 'accs'},
    'вокруг': {'gent', 'accs'},
    'около': {'gent', 'accs'},
    'через': {'accs', 'ablt'},
    'мимо': {'gent', 'accs'},
    'благодаря': {'ablt', 'datv'},
    'навстречу': {'datv', 'accs'},
    'наперекор': {'datv', 'accs'},
    'против': {'gent', 'accs'},
    'вдоль': {'gent', 'accs'},
    'сверх': {'gent', 'accs'},
    'позади': {'gent', 'accs'},
    'прежде': {'gent', 'accs'}
}


In [160]:
with open('/Users/katyamazurina/Desktop/texts/turgenev-nov.txt', 'r', encoding='Windows 1251') as file:
    text = file.read().lower()

In [167]:
text_tokenized = word_tokenize(text)

cases = defaultdict(lambda: defaultdict(int)) # словарь для подсчета падежей

for i in range(len(text_tokenized)-1): # проходим по всем токенам, кроме последнего (тк мы проверяем только текущий и следующий токены)
    token = text_tokenized[i]
    if morph.parse(token)[0].tag.POS == 'PREP' and token in preps: # проверка, является ли текущий токен предлогом 
        if morph.parse(text_tokenized[i + 1])[0].tag.case in preps[token]: # если предыдущая проверка выполняется, переходим к следующему токену и проверяем его падеж
            cases[token][morph.parse(text_tokenized[i + 1])[0].tag.case] += 1 # добавляем в словарь, если проверка выполняется

for prep, case_counts in cases.items():
    print(f"\n После предлога '{prep}' встречаются падежи: ")
    for case, count in case_counts.items():
        print(f'{case}: {count} раз')


 После предлога 'в' встречаются падежи: 
loct: 584 раз
accs: 419 раз

 После предлога 'с' встречаются падежи: 
ablt: 539 раз
accs: 22 раз
gent: 198 раз

 После предлога 'к' встречаются падежи: 
datv: 316 раз
loct: 33 раз

 После предлога 'на' встречаются падежи: 
accs: 347 раз
loct: 211 раз

 После предлога 'за' встречаются падежи: 
ablt: 80 раз
accs: 65 раз

 После предлога 'над' встречаются падежи: 
accs: 2 раз
ablt: 19 раз

 После предлога 'под' встречаются падежи: 
ablt: 29 раз
accs: 15 раз

 После предлога 'о' встречаются падежи: 
loct: 131 раз
accs: 13 раз

 После предлога 'по' встречаются падежи: 
datv: 161 раз
loct: 27 раз
accs: 16 раз

 После предлога 'об' встречаются падежи: 
accs: 10 раз
loct: 38 раз

 После предлога 'из-за' встречаются падежи: 
gent: 10 раз
accs: 1 раз

 После предлога 'против' встречаются падежи: 
accs: 3 раз
gent: 7 раз

 После предлога 'перед' встречаются падежи: 
ablt: 69 раз
accs: 1 раз

 После предлога 'через' встречаются падежи: 
accs: 33 раз

 Посл

#### Задача 3 (5 баллов). 

Представим, что у вас есть файл с разборами conllu (можете взять любой, например, [тут](https://github.com/dialogue-evaluation/GramEval2020)). Нужно просмотреть все примеры предложений с тегом dislocated и тегом discourse: напишите скрипт, который будет читать файл, находить все такие предложения и печатать: 1) сам текст предложения 2) слово, имеющее искомый тег. Если тег не был найден в файле, нужно об этом сообщить. Постарайтесь оформить вывод таким образом, чтобы это было удобно читать. 

In [8]:
import pyconll

In [9]:
text_conllu =  pyconll.load_from_file('/Users/katyamazurina/Desktop/texts/GramEval2020-Taiga-social-train.conllu')

In [168]:
found_tag = False

for sentence in text_conllu:
    sentence_text = " ".join([token.form for token in sentence]) 

    for token in sentence:
        if 'dislocated' in token.deprel:
            print(f"\nТокен '{token.form.upper()}' с тегом dislocated в предложении <{sentence_text}> ")
            found_tag = True

        elif 'discourse' in token.deprel:
            print(f"\nТокен '{token.form.upper()}' с тегом  discourse в предложении <{sentence_text}> ")
            found_tag = True

if not found_tag:
    print(f"Теги 'dislocated' и 'discourse' не найдены")


Токен ')' с тегом  discourse в предложении <Чудесная бутылочка в наличии и под заказ )> 

Токен '@' с тегом  discourse в предложении <Вот в такой компании провел вечернюю прогулку @> 

Токен 'ЧТО' с тегом dislocated в предложении <Всеволод , Вы же лицо " Гражданской Силы " , Вам что стыдно быть похожим на депутата главенствующей партии РФСтыдно !> 

Токен 'ТО' с тегом  discourse в предложении <Стоит ли электорату КПРФ беспокоить " Гену - пчеловода " перед выборами - то там поддержки коммунистов совсем нет !> 

Токен '")))' с тегом  discourse в предложении <" Кто голосует за ПАРНАС - Тому любая баба даст ! " ")))> 

Токен ':)' с тегом  discourse в предложении <Явлинский : одно из главных достижений яблока в 2012 -- избрания Каца в Щ :)> 

Токен '@YABLOKO' с тегом dislocated в предложении <@xxxxxx @yabloko если и @yabloko Вы так будете руководить , основываясь на бездоказательных выводах , то и за вас теперь не буду голосовать> 

Токен '))))' с тегом  discourse в предложении <Аксенов пр

#### Задача 4 (5 баллов).

Возьмите любой достаточно длинный (лучше новостной) текст. Любым известным инструментом извлеките именованные сущности из этого текста и выведите их списком по категориям (т.е. персоны вместе, локации вместе, организации вместе). 

In [169]:
import natasha

from natasha import (
    Segmenter,
    NewsEmbedding,
    NewsNERTagger,
    Doc
)

segmenter = Segmenter()  
emb = NewsEmbedding() 
ner_tagger = NewsNERTagger(emb)

In [170]:
with open('/Users/katyamazurina/Desktop/texts/news.txt', 'r', encoding='maccyrillic') as file:
    text = file.read()    

In [171]:
doc = Doc(text)
doc.segment(segmenter) 
doc.tag_ner(ner_tagger) 

LOC = [] 
PER = []
ORG = []

for span in doc.spans:
    if span.type == 'LOC':
        LOC.append(span.text)
    elif span.type == 'PER':
        PER.append(span.text)
    elif span.type == 'ORG':
        ORG.append(span.text)
   
print(f'LOC-сущности ({len(LOC)}): {LOC}')
print(f'PER-сущности ({len(PER)}): {PER}')
print(f'ORG-сущности ({len(ORG)}): {ORG}')

LOC-сущности (18): ['США', 'Украине', 'Израилю', 'Штатов', 'США', 'США', 'Вашингтона', 'Киеве', 'РФ', 'Израиль', 'США', 'Газа', 'Израилю', 'Ирану', 'США', 'Украине', 'Украины', 'России']
PER-сущности (14): ['Дональд Трамп', 'Трамп', 'Дональда Трампа', 'Джо Байдена', 'Байден', 'Шелби Магид', 'Трампа', 'Биньямин Нетаньяху', 'Трампа', 'Джо Байденом', 'Дональд Трамп', 'Трамп', 'Трампа', 'Владимир Зеленский']
ORG-сущности (6): ['Bloomberg', 'Евразийского центра', 'Атлантического совета', 'Bloomberg', 'The Wall Street Journal', 'НАТО']


#### Задача на бонусные 5 баллов:

Сравните качество несколькиз разных морфопарсеров для любого языка, где их больше одного. Разберите этими морфопарсерами один и тот же текст, если они все разбирают в UD, можете вывести автоматически расхождения, в противном случае просмотрите глазами на наличие ошибок (текст, конечно, слишком большой лучше не брать). 

Без письменных выводов-комментариев не засчитывается. 

In [ ]:
# your code here